In [14]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from langchain.output_parsers import PydanticOutputParser, StructuredOutputParser, ResponseSchema
from pydantic import BaseModel, Field

load_dotenv()
google_api_key=os.getenv('GOOGLE_API_KEY')

llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0)

In [4]:
#1. StrOutputParser:
prompt= PromptTemplate.from_template('Summarize the text: {text}')
parser= StrOutputParser()

chain= prompt | llm | parser

response1= chain.invoke({'text':'''LangChain is an open-source framework designed to simplify the development, production, and deployment of applications 
powered by large language models (LLMs) like GPT-4. It provides a standardized toolkit that allows developers to "chain" together different 
components—such as data sources, computation tools, and various AI models—to build complex AI workflows'''})

print(response1)

LangChain is an open-source framework designed to simplify the development and deployment of applications powered by large language models (LLMs). It provides a standardized toolkit that allows developers to "chain" together different components like data sources, computation tools, and AI models to build complex AI workflows.


In [6]:
#2. CommaSeparatedListOutputParser:
prompt= PromptTemplate.from_template('Extract and list only the technical skills from the resume text: {text}')
parser= CommaSeparatedListOutputParser()

chain= prompt | llm | parser

response2= chain.invoke({'text':'''
Mandatory Skills:
Strong expertise in Generative AI and agentic AI technologies.
Proven experience in architectural design and implementation.
Excellent problem-solving and analytical skills.
Strong communication and interpersonal skills.

Preferred Skills:
Experience with cloud architecture and services.
Familiarity with machine learning frameworks and tools.
Knowledge of software development methodologies.
Experience in leading architectural projects in a collaborative environment.'''})

print(response2)

['Here are the technical skills from the resume text:', '*   Generative AI and agentic AI technologies', '*   Architectural design and implementation', '*   Cloud architecture and services', '*   Machine learning frameworks and tools']


In [15]:
#3. PydanticOutputParser:
class movie_review(BaseModel):
    sentiment: str= Field(description= 'Tone/ emotion: negative, neutral, positive')
    rating: int = Field(description= 'Ratings given from 1 to 5')
    key_complaint: str = Field(description= 'Issues reported about the movie')

parser= PydanticOutputParser(pydantic_object= movie_review)

prompt= PromptTemplate(
    template= 'Given review text, return sentiment, rating(1-5), key complaint: {text} \n {format_instructions}',
    input_variables= ['text'],
    partial_variables= {'format_instructions':parser.get_format_instructions()}
)

chain= prompt | llm | parser

response3= chain.invoke({'text':'''
Review 2: This movie was a complete waste of time. The story made no sense, and the dialogues were cringe. The comedy scenes felt forced and unnecessary.
The villain character was badly written and didn’t feel threatening at all. I regret watching it in the theatre.'''})

print(response3)

sentiment='negative' rating=1 key_complaint='Poor writing and execution of the story, dialogues, comedy, and villain character.'


In [20]:
#4. StructuredOutputParser:
schemas= [
    ResponseSchema(name= 'name', description= 'Name of the movie'),
    ResponseSchema(name= 'genre', description= 'Movie categories based on story'),
    ResponseSchema(name= 'reason', description= 'Reason for choosing the movie'),
    ResponseSchema(name= 'rating', description= 'Ratings given from 1 to 5')
]

parser= StructuredOutputParser.from_response_schemas(schemas)

prompt= PromptTemplate(
    template= '''Recommend 1 latest South Indian movie with name, genre, reason, rating: \n {format_instructions}. 
    Return ONLY valid JSON. Do not include bullet points or markdown.''',
    input_variables= [],
    partial_variables= {'format_instructions':parser.get_format_instructions()}
)

chain= prompt | llm | parser

response4= chain.invoke({})
print(response4)

{'name': 'Aadujeevitham (The Goat Life)', 'genre': 'Survival Drama', 'reason': 'This film is a powerful and emotionally gripping survival drama based on a true story, featuring a career-best performance by Prithviraj Sukumaran. It has received widespread critical acclaim for its direction, cinematography, and the raw portrayal of human resilience against extreme adversity.', 'rating': '4.5'}
